## 3. Train Validation Test----------------------------------------------------------

In [ ]:
# import the libraries
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Train-Val-Test 60/20/20
# splitting data target and feature variables
fct_cleaned = fct.dropna(subset=['Cover_Type'])

# Drop the string-based 'Wilderness_Area' and 'Soil_Type' columns
cols_to_drop = ['Wilderness_Area', 'Soil_Type']
fct_cleaned = fct_cleaned.drop(columns=[col for col in cols_to_drop if col in fct_cleaned.columns])

X = fct_cleaned.drop('Cover_Type', axis=1)
y = fct_cleaned['Cover_Type']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    stratify=y_train_val, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")

Train: (348606, 54)
Val: (116203, 54)
Test: (116203, 54)


In [ ]:
# Preprocessing
cat_cols = []  # ["Wilderness_Area", "Soil_Type"] in case they are categorical columns
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = "f1_macro" # more suitable for imbalanced multiclass

## 4. Preprocessing----------------------------------------

### Preprocessing for future final conclusion

In [ ]:
# import neccessary libraries
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

RANDOM_STATE = 42

In [ ]:
# Help/support functions
def eval_on_val(model_name, fitted_estimator, X_val, y_val):
    """Return validation metrics in a dict."""
    pred = fitted_estimator.predict(X_val)
    return {
        "Model": model_name,
        "Val_Accuracy": accuracy_score(y_val, pred),
        "Val_BalancedAcc": balanced_accuracy_score(y_val, pred),
        "Val_F1_Macro": f1_score(y_val, pred, average="macro")
    }

def print_final_reports(best_name, best_model, X_val, y_val, X_test, y_test):
    print("\n" + "="*70)
    print(f"BEST MODEL: {best_name}")
    print("="*70)

    # Validation
    val_pred = best_model.predict(X_val)
    print("\n[VALIDATION] Classification Report")
    print(classification_report(y_val, val_pred))
    print("[VALIDATION] Confusion Matrix")
    print(confusion_matrix(y_val, val_pred))

    # Test (ONLY ONCE, at the end)
    test_pred = best_model.predict(X_test)
    print("\n[TEST] Classification Report")
    print(classification_report(y_test, test_pred))
    print("[TEST] Confusion Matrix")
    print(confusion_matrix(y_test, test_pred))

# store best estimator from each branch
results = []
fitted_models = {}

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Convert X_train.columns to a list to avoid potential issues with ColumnTransformer accepting a pandas Index object
numeric_features = list(X_train.columns)   # ALL columns are numeric

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features)
    ],
    remainder="drop"
)